# EEDI and OpinionQA Quantile Reproduction

This notebook reproduces the appendix EEDI and OpinionQA paper figures directly from the prepared artifacts in `data/eedi/` and `data/opinionqa/`. It skips the upstream data-cleaning pipeline and focuses on the final figure-generation path used in the paper.


## Repository Root Setup

In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    direct = candidate / 'figures'
    nested = candidate / 'paper_reproduction' / 'eedi_opinionqa_quantile'
    if (candidate / 'README.md').exists() and direct.exists() and candidate.name == 'eedi_opinionqa_quantile':
        NOTEBOOK_DIR = candidate
        break
    if nested.exists():
        NOTEBOOK_DIR = nested
        break
if NOTEBOOK_DIR is None:
    raise RuntimeError('Could not locate paper_reproduction/eedi_opinionqa_quantile.')

ROOT = NOTEBOOK_DIR.parent
os.chdir(ROOT)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

REPRO_DIR = ROOT / 'eedi_opinionqa_quantile'
FIGURES_DIR = REPRO_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Reproduction root: {ROOT}')
print(f'Figure output directory: {FIGURES_DIR}')


In [ ]:
import copy
import json
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display


## EEDI

The cells below load the prepared EEDI survey and simulator artifacts and reproduce the paper's Figure 13 quantile-fidelity plot with absolute loss and simulation budget `k = 50`.


In [ ]:
good_questions = np.load(ROOT / 'data/eedi/good_questions.npy', allow_pickle=True)
with open(ROOT / 'data/eedi/good_questions_statement.json', 'r') as f:
    good_questions_statement = json.load(f)
with open(ROOT / 'data/eedi/good_questions_answer.json', 'r') as f:
    good_questions_answer = json.load(f)
with open(ROOT / 'data/eedi/surveys.pkl', 'rb') as f:
    surveys = pickle.load(f)

real_answers = {}
for question_id in good_questions:
    vals_i = copy.deepcopy(surveys[question_id]['IsCorrect'].values)
    real_answers[question_id] = vals_i

synthetic_answers_all = {}
with open(ROOT / 'data/eedi/synthetic answers/iscorrect/synthetic_answers_iscorrect_3.5turbo.pkl', 'rb') as f:
    synthetic_answers_all['gpt-3.5-turbo'] = pickle.load(f)
with open(ROOT / 'data/eedi/synthetic answers/iscorrect/synthetic_answers_iscorrect_4omini.pkl', 'rb') as f:
    synthetic_answers_all['gpt-4o-mini'] = pickle.load(f)
with open(ROOT / 'data/eedi/synthetic answers/iscorrect/synthetic_answers_iscorrect_4o.pkl', 'rb') as f:
    synthetic_answers_all['gpt-4o'] = pickle.load(f)
with open(ROOT / 'data/eedi/synthetic answers/iscorrect/synthetic_answers_iscorrect_claude.pkl', 'rb') as f:
    synthetic_answers_all['claude-3-5-haiku'] = pickle.load(f)
with open(ROOT / 'data/eedi/synthetic answers/iscorrect/synthetic_answers_iscorrect_Llama-3.3-70B-Instruct-Turbo.pkl', 'rb') as f:
    synthetic_answers_all['Llama-3.3-70B'] = pickle.load(f)
with open(ROOT / 'data/eedi/synthetic answers/iscorrect/synthetic_answers_iscorrect_Mistral-7B-Instruct-v0.3.pkl', 'rb') as f:
    synthetic_answers_all['Mistral-7B'] = pickle.load(f)
with open(ROOT / 'data/eedi/synthetic answers/iscorrect/synthetic_answers_iscorrect_DeepSeek-V3.pkl', 'rb') as f:
    synthetic_answers_all['DeepSeek-V3'] = pickle.load(f)
with open(ROOT / 'data/eedi/synthetic answers/iscorrect/synthetic_answers_iscorrect_random.pkl', 'rb') as f:
    synthetic_answers_all['random'] = pickle.load(f)

print(f'Loaded EEDI questions: {len(good_questions)}')
print(f'Loaded EEDI simulators: {sorted(synthetic_answers_all)}')


In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

# ============================================================
# (0) Small utilities: coercion + subsampling (human & synthetic)
# ============================================================

def _coerce_1d_float_array(x):
    """
    Convert array-like / pd.Series to 1D float np.ndarray (keeps NaNs for now).
    """
    if hasattr(x, "values"):
        x = x.values
    return np.asarray(x, dtype=float).ravel()

def _finite_1d_float_array(x):
    """
    Convert to 1D float np.ndarray and drop NaNs/infs.
    """
    arr = _coerce_1d_float_array(x)
    return arr[np.isfinite(arr)]

def _subsample_array(x, n_target=None, rng=None, replace=False):
    """
    Generic subsample helper:
    - coerces to float, drops non-finite
    - if n_target None or <=0 => returns all finite
    """
    if rng is None:
        rng = np.random.default_rng()

    y = _finite_1d_float_array(x)
    n_avail = int(y.size)
    if n_avail == 0:
        return np.asarray([], dtype=float), 0

    if (n_target is None) or (int(n_target) <= 0):
        return y, n_avail

    n_use = int(min(int(n_target), n_avail))
    if (not replace) and (n_use == n_avail):
        return y, n_avail

    idx = rng.choice(n_avail, size=n_use, replace=bool(replace))
    return y[idx], n_use

def _subsample_human(yh, n_target=None, rng=None, replace=False):
    """
    Alias kept for backwards compatibility with your earlier code.
    """
    return _subsample_array(yh, n_target=n_target, rng=rng, replace=replace)

def _subsample_synth(ys, k_target, rng=None, replace=False, first_k=False):
    """
    Subsample synthetic answers safely.
    If first_k=True, take first k (after finite-filtering). Otherwise random sample.
    """
    if rng is None:
        rng = np.random.default_rng()

    x = _finite_1d_float_array(ys)
    if x.size == 0:
        return np.asarray([], dtype=float), 0

    k_use = int(min(int(k_target), int(x.size)))
    if k_use <= 0:
        return np.asarray([], dtype=float), 0

    if first_k or (k_use == x.size and not replace):
        return x[:k_use], k_use

    idx = rng.choice(x.size, size=k_use, replace=bool(replace))
    return x[idx], k_use


# ============================================================
# (1) Gamma schedules g(n): swappable and clipped
# ============================================================

def gamma_1_minus_inv_sqrtn(n_eff: int, eps: float = 1e-6) -> float:
    """
    g(n) = 1 - 1/sqrt(n), clipped to (eps, 1-eps).
    """
    n_eff = int(n_eff)
    if n_eff <= 0:
        return np.nan
    g = 1.0 - (n_eff ** (-0.5))
    return float(np.clip(g, eps, 1.0 - eps))

def gamma_1_minus_inv_n(n_eff: int, eps: float = 1e-6) -> float:
    """
    g(n) = 1 - 1/n, clipped to (eps, 1-eps).
    """
    n_eff = int(n_eff)
    if n_eff <= 0:
        return np.nan
    g = 1.0 - (1.0 / n_eff)
    return float(np.clip(g, eps, 1.0 - eps))

def _resolve_gamma_j(n_eff: int, gamma_fixed: float, use_adaptive_gamma: bool, gamma_fn, gamma_fn_kwargs):
    """
    If adaptive: gamma_j = gamma_fn(n_eff, **kwargs).
    Else: gamma_j = gamma_fixed.
    """
    if not use_adaptive_gamma:
        return float(gamma_fixed)
    return float(gamma_fn(int(n_eff), **(gamma_fn_kwargs or {})))


# ============================================================
# (2) Losses on Bernoulli means
# ============================================================

def kl_bern(p, q, q_eps=1e-12):
    q = float(min(max(q, q_eps), 1.0 - q_eps))
    if p <= 0.0:  return -math.log(1.0 - q)
    if p >= 1.0:  return -math.log(q)
    return p*math.log(p/q) + (1-p)*math.log((1-p)/(1-q))

def tv_bern(p, q):
    return abs(float(p) - float(q))

def hellinger2_bern(p, q, q_eps=1e-12):
    q = float(min(max(q, q_eps), 1.0 - q_eps))
    return 0.5*((math.sqrt(p)-math.sqrt(q))**2 + (math.sqrt(1-p)-math.sqrt(1-q))**2)

def l2_bern(p, q):
    d = float(p) - float(q)
    return d*d

def make_loss(kind="kl", custom_loss=None):
    if kind == "kl":         return lambda p, q: kl_bern(p, q)
    if kind == "tv":         return lambda p, q: tv_bern(p, q)
    if kind == "hellinger":  return lambda p, q: hellinger2_bern(p, q)
    if kind == "l2":         return lambda p, q: l2_bern(p, q)
    if kind == "custom":
        if custom_loss is None:
            raise ValueError("Provide custom_loss when kind='custom'.")
        return custom_loss
    raise ValueError(f"Unknown loss kind: {kind}")


# ============================================================
# (3) KL-ball around phat: endpoints in p
# ============================================================

def _D_kl_ph_to_p(ph, p):
    ph = float(ph); p = float(p)
    if ph <= 0.0:
        if p >= 1.0: return float("inf")
        return -math.log(max(1.0 - p, 1e-300))
    if ph >= 1.0:
        if p <= 0.0: return float("inf")
        return -math.log(max(p, 1e-300))
    if p <= 0.0 or p >= 1.0:
        return float("inf")
    return ph*math.log(ph/p) + (1-ph)*math.log((1-ph)/(1-p))

def _bisect_target(fun, a, b, target, tol=1e-10, it=100):
    fa, fb = fun(a)-target, fun(b)-target
    if fa*fb > 0:
        return a if abs(fa) < abs(fb) else b
    lo, hi = a, b
    for _ in range(it):
        mid = 0.5*(lo+hi)
        fm  = fun(mid) - target
        if fa*fm <= 0:
            hi, fb = mid, fm
        else:
            lo, fa = mid, fm
        if hi - lo < tol:
            break
    return 0.5*(lo+hi)

def kl_ball_endpoints_in_p(ph, r):
    ph = float(ph); r = float(max(r, 0.0))
    if r == 0.0:
        return ph, ph
    if ph <= 0.0:
        return 0.0, 1.0 - math.exp(-r)
    if ph >= 1.0:
        return math.exp(-r), 1.0
    pL = _bisect_target(lambda p: _D_kl_ph_to_p(ph, p), 0.0, ph, r)
    pU = _bisect_target(lambda p: _D_kl_ph_to_p(ph, p), ph, 1.0, r)
    return pL, pU


# ============================================================
# (4) Upper pseudo-discrepancy: Δ̂ = sup_{p in KL-ball} L(p, q̂)
# ============================================================

def delta_hat_one(
    phat,
    qhat_raw,
    n_h,
    k,
    beta=0.5,
    loss_kind="kl",
    maximize="endpoints",
    grid_points=401,
    clip_c=0.05,
):
    if k is None or int(k) <= 0:
        raise ValueError("k must be > 0.")

    # synthetic-only clipping
    eps_k = float(clip_c) / float(k)
    qhat  = float(min(max(float(qhat_raw), eps_k), 1.0 - eps_k))

    # Chernoff/KL radius
    r = math.log(2.0 / float(beta)) / max(1, int(n_h))

    # KL-ball endpoints
    pL, pU = kl_ball_endpoints_in_p(phat, r)

    L = make_loss(loss_kind)
    if maximize == "endpoints":
        d = max(L(pL, qhat), L(pU, qhat))
    else:
        xs = np.linspace(pL, pU, int(grid_points))
        d = float(max(L(x, qhat) for x in xs))

    return d, {"pL": pL, "pU": pU, "qhat": qhat, "eps_k": eps_k, "r": r}


# ============================================================
# (5) Empirical quantile curve for plotting
# ============================================================

def empirical_quantile_curve(x, probs):
    """
    Left-continuous empirical quantile function at probabilities in [0,1].
    Matches the typical 'inf{t: Fhat(t) >= p}' convention.
    """
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.full_like(np.asarray(probs, float), np.nan, dtype=float)
    xs = np.sort(x)
    p = np.asarray(probs, float)
    p = np.clip(p, 0.0, 1.0)
    # left-continuous: q(p) = xs[ceil(p*n)-1], with clamp
    idx = np.ceil(p * xs.size).astype(int) - 1
    idx = np.clip(idx, 0, xs.size - 1)
    return xs[idx]


# ============================================================
# (6) Compute deltas for EEDI dataset: output df_long + gamma_summary
# ============================================================

def compute_deltas_eedi_klball(
    good_q,
    real_answers,
    synthetic_answers_all,
    *,
    k=50,
    n_target=None,
    gamma_fixed=0.9,
    use_adaptive_gamma=True,
    gamma_fn=gamma_1_minus_inv_sqrtn,   # or gamma_1_minus_inv_n
    gamma_fn_kwargs=None,
    loss_kind="tv",
    maximize="endpoints",
    grid_points=401,
    clip_c=0.05,
    seed=0,
    human_replace=False,
    synth_first_k=False,                # if True, take first k; else random k
):
    """
    Inputs (EEDI):
      - good_q: iterable of question ids (np array or list)
      - real_answers: dict[qid] -> array/Series of human IsCorrect (0/1 or bool)
      - synthetic_answers_all: dict[model] -> dict[qid] -> array/Series (0/1 or bool)

    Output:
      - df_long with columns:
          ['qid','sim','delta','n_eff','k_used','gamma_j','beta_j','gamma_mode']
      - gamma_summary with bar_gamma_overall, by-sim means, counts
    """
    gamma_fn_kwargs = gamma_fn_kwargs or {}
    rng_master = np.random.default_rng(seed)
    rows = []
    gamma_mode = "adaptive" if use_adaptive_gamma else "fixed"

    for sim_name, synth_map in synthetic_answers_all.items():
        for qid in good_q:
            if qid not in real_answers or qid not in synth_map:
                continue

            local_seed = int(rng_master.integers(0, 2**32 - 1))
            rng = np.random.default_rng(local_seed)

            # human subsample
            yh_sub, n_eff = _subsample_human(
                real_answers[qid], n_target=n_target, rng=rng, replace=human_replace
            )
            if n_eff <= 0:
                continue

            # synthetic subsample
            xs_use, k_used = _subsample_synth(
                synth_map[qid], k_target=k, rng=rng, replace=False, first_k=bool(synth_first_k)
            )
            if k_used <= 0:
                continue

            phat = float(np.mean(yh_sub))
            qhatraw = float(np.mean(xs_use))

            gamma_j = _resolve_gamma_j(
                n_eff=n_eff,
                gamma_fixed=float(gamma_fixed),
                use_adaptive_gamma=bool(use_adaptive_gamma),
                gamma_fn=gamma_fn,
                gamma_fn_kwargs=gamma_fn_kwargs,
            )
            if (not np.isfinite(gamma_j)) or (gamma_j <= 0.0) or (gamma_j >= 1.0):
                continue

            # map coverage gamma_j -> tail prob beta_j
            beta_j = float(np.clip(1.0 - gamma_j, 1e-12, 1.0))

            d, info = delta_hat_one(
                phat=phat,
                qhat_raw=qhatraw,
                n_h=int(n_eff),
                k=int(k_used),
                beta=float(beta_j),
                loss_kind=loss_kind,
                maximize=maximize,
                grid_points=grid_points,
                clip_c=clip_c,
            )
            if not np.isfinite(d):
                continue

            rows.append({
                "qid": qid,
                "sim": sim_name,
                "delta": float(d),
                "n_eff": int(n_eff),
                "k_used": int(k_used),
                "gamma_j": float(gamma_j),
                "beta_j": float(beta_j),
                "gamma_mode": gamma_mode,
            })

    df = pd.DataFrame(rows, columns=[
        "qid","sim","delta","n_eff","k_used","gamma_j","beta_j","gamma_mode"
    ])

    if len(df) > 0:
        gamma_summary = {
            "gamma_mode": gamma_mode,
            "bar_gamma_overall": float(df["gamma_j"].mean()),
            "n_rows_overall": int(len(df)),
            "bar_gamma_by_sim": {str(k): float(v) for k, v in df.groupby("sim")["gamma_j"].mean().items()},
            "n_rows_by_sim": {str(k): int(v) for k, v in df.groupby("sim").size().items()},
        }
    else:
        gamma_summary = {
            "gamma_mode": gamma_mode,
            "bar_gamma_overall": np.nan,
            "n_rows_overall": 0,
            "bar_gamma_by_sim": {},
            "n_rows_by_sim": {},
        }

    return df, gamma_summary


# ============================================================
# (7) Plot paper-style quantile fidelity curve: alpha -> Vhat(alpha)
#     tau* = (1 - bar_gamma) + bar_gamma * tau
# ============================================================

def plot_upper_gamma_indexed_multi_adaptive(
    deltas_data,                   # DataFrame (preferred) or dict[sim]->deltas
    gamma_summary=None,
    bar_gamma=None,
    sim_names=None,
    tau_grid=None,
    color_map=None,
    linewidth=2.5,
    linestyle="-",
    legend=True,
    xlabel=r"$\tau$",
    ylabel=None,
    title=None,
    figsize=(7.5, 5),
):
    """
    Asymptotic theorem index map:
        tau ↦ V̂_m( (1 - bar_gamma) + bar_gamma * tau )
    where V̂_m is the empirical quantile function of {delta_j} across qids.
    """
    # ---- accept EEDI long DF directly
    if isinstance(deltas_data, pd.DataFrame):
        if ("sim" not in deltas_data.columns) or ("delta" not in deltas_data.columns):
            raise ValueError("DataFrame must have columns ['sim','delta'].")
        deltas_dict = {
            str(sim): np.asarray(g["delta"], float)
            for sim, g in deltas_data.groupby("sim", sort=True)
        }
    elif isinstance(deltas_data, dict):
        deltas_dict = {k: np.asarray(v, float) for k, v in deltas_data.items()}
    else:
        raise TypeError("deltas_data must be a DataFrame or dict {sim: deltas}.")

    # ---- choose bar_gamma
    if bar_gamma is None:
        if gamma_summary is None:
            raise ValueError("Provide bar_gamma or gamma_summary (returned by compute_deltas_eedi_klball).")
        bar_gamma = float(gamma_summary.get("bar_gamma_overall", np.nan))

    if (not np.isfinite(bar_gamma)) or (bar_gamma <= 0.0) or (bar_gamma >= 1.0):
        raise ValueError(f"Invalid bar_gamma={bar_gamma}. Expected in (0,1).")

    # ---- grid
    if sim_names is None:
        sim_names = list(deltas_dict.keys())
    if tau_grid is None:
        tau_grid = np.linspace(0.0, 1.0, 201)
    tau_grid = np.asarray(tau_grid, float)

    tau_star = np.clip((1.0 - bar_gamma) + bar_gamma * tau_grid, 0.0, 1.0)

    fig, ax = plt.subplots(figsize=figsize)
    out = {}

    for name in sim_names:
        if name not in deltas_dict:
            continue

        deltas = np.asarray(deltas_dict[name], float)
        deltas = deltas[np.isfinite(deltas)]
        if deltas.size == 0:
            continue

        V_upper = empirical_quantile_curve(deltas, tau_star)

        color = None if color_map is None else color_map.get(name, None)
        plt.plot(
            tau_grid,
            V_upper,
            linestyle=linestyle,
            linewidth=linewidth,
            color=color,
            label=str(name),
        )

        out[name] = {
            "tau": tau_grid,
            "tau_star": tau_star,
            "V_upper": V_upper,
            "bar_gamma": float(bar_gamma),
        }

    plt.xlabel(xlabel)
    if ylabel is None:
        ylabel = r"$\widehat V_m\!\left((1-\bar\gamma)+\bar\gamma\,\tau\right)$"
    plt.ylabel(ylabel)

    if title is None:
        title = rf"Upper curve with index shift, $\bar\gamma$ = {bar_gamma:.3f}"
    plt.title(title)

    if legend:
        plt.legend(fontsize=10)

    plt.tight_layout()
    plt.show()
    return out


# ============================================================
# C) Paper-style quantile fidelity plot: alpha -> Vhat(alpha)
# ============================================================

def plot_quantile_fidelity_profiles(
    df_long, *,
    sim_names=None, alpha_grid=None, color_map=None,
    linewidth=2.0, linestyle="-", legend=True,
    xlabel=r"$\alpha$", ylabel=r"$\widehat V_\ell(\alpha)$", title=None,
    figsize=(7.8, 5.2),
):
    if sim_names is None:
        sim_names = sorted(df_long["sim"].unique().tolist())

    if alpha_grid is None:
        alpha_grid = np.linspace(0.0, 1.0, 201)
    alpha_grid = np.asarray(alpha_grid, float)

    fig, ax = plt.subplots(figsize=figsize)
    out = {}

    for sim in sim_names:
        sub = df_long[df_long["sim"] == sim]
        deltas = np.asarray(sub["delta"], float)
        deltas = deltas[np.isfinite(deltas)]
        if deltas.size == 0:
            continue

        V = empirical_quantile_curve(deltas, alpha_grid)
        color = None if color_map is None else color_map.get(sim, None)
        ax.plot(alpha_grid, V, linestyle=linestyle, linewidth=linewidth, color=color, label=str(sim))
        out[str(sim)] = {"alpha": alpha_grid, "V_hat": V}

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title)
    if legend:
        ax.legend(fontsize=9, ncol=2, framealpha=0.3)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()
    return fig, out


### Final EEDI Plot

In [ ]:
EEDI_PAPER_MODEL_ORDER = ['gpt-3.5-turbo', 'gpt-4o', 'gpt-4o-mini', 'claude-3-5-haiku', 'Llama-3.3-70B', 'Mistral-7B', 'DeepSeek-V3', 'random']

eedi_df_long, eedi_gamma_summary = compute_deltas_eedi_klball(
    good_q=good_questions,
    real_answers=real_answers,
    synthetic_answers_all=synthetic_answers_all,
    k=50,
    n_target=None,
    use_adaptive_gamma=True,
    gamma_fn=gamma_1_minus_inv_sqrtn,
    loss_kind='tv',
    seed=0,
    synth_first_k=False,
)

eedi_fig, _ = plot_quantile_fidelity_profiles(
    eedi_df_long,
    sim_names=EEDI_PAPER_MODEL_ORDER,
    title='EEDI quantile fidelity profiles',
)

eedi_fig_path = FIGURES_DIR / 'eedi_quantile_fidelity_profiles.png'
eedi_fig.savefig(eedi_fig_path, dpi=200, bbox_inches='tight')
plt.close(eedi_fig)
print(f'Saved EEDI figure to: {eedi_fig_path}')
display(Image(filename=str(eedi_fig_path)))


![Figure 13 reproduction](figures/eedi_quantile_fidelity_profiles.png)


## OpinionQA

The cells below load the prepared OpinionQA artifacts and reproduce the paper's Figure 14 quantile-fidelity plot with total variation loss and simulation budget `k = 100`.


In [ ]:
good_questions = np.load(ROOT / 'data/opinionqa/good_questions.npy', allow_pickle=True)
Qs = pd.read_csv(ROOT / 'data/opinionqa/Qs_to_use.csv')
with open(ROOT / 'data/opinionqa/surveys.pkl', 'rb') as f:
    surveys = pickle.load(f)

real_answers = {}
for qid in good_questions:
    qid = str(qid)
    vals_i = surveys[qid]['RESPONSE_NUMERIC'].values
    real_answers[qid] = vals_i

synthetic_answers_all = {}
with open(ROOT / 'data/opinionqa/synthetic answers/clean/synthetic_answers_clean_3.5turbo.pkl', 'rb') as f:
    synthetic_answers_all['gpt-3.5-turbo'] = pickle.load(f)
with open(ROOT / 'data/opinionqa/synthetic answers/clean/synthetic_answers_clean_4omini.pkl', 'rb') as f:
    synthetic_answers_all['gpt-4o-mini'] = pickle.load(f)
with open(ROOT / 'data/opinionqa/synthetic answers/clean/synthetic_answers_clean_4o.pkl', 'rb') as f:
    synthetic_answers_all['gpt-4o'] = pickle.load(f)
with open(ROOT / 'data/opinionqa/synthetic answers/clean/synthetic_answers_clean_claude.pkl', 'rb') as f:
    synthetic_answers_all['claude-3-5-haiku'] = pickle.load(f)
with open(ROOT / 'data/opinionqa/synthetic answers/clean/synthetic_answers_clean_Llama-3.3-70B-Instruct-Turbo.pkl', 'rb') as f:
    synthetic_answers_all['Llama-3.3-70B'] = pickle.load(f)
with open(ROOT / 'data/opinionqa/synthetic answers/clean/synthetic_answers_clean_Mistral-7B-Instruct-v0.3.pkl', 'rb') as f:
    synthetic_answers_all['Mistral-7B'] = pickle.load(f)
with open(ROOT / 'data/opinionqa/synthetic answers/clean/synthetic_answers_clean_DeepSeek-V3.pkl', 'rb') as f:
    synthetic_answers_all['DeepSeek-V3'] = pickle.load(f)
with open(ROOT / 'data/opinionqa/synthetic answers/clean/synthetic_answers_clean_random.pkl', 'rb') as f:
    synthetic_answers_all['random'] = pickle.load(f)

synthetic_answers_all = {sim: {str(k): v for k, v in sim_map.items()} for sim, sim_map in synthetic_answers_all.items()}
levels = np.array([-1.0, -1.0/3.0, 0.0, 1.0/3.0, 1.0], dtype=float)

print("OpinionQA support:", levels)
print(f'Loaded OpinionQA questions: {len(good_questions)}')
print(f'Loaded OpinionQA simulators: {sorted(synthetic_answers_all)}')


In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
from math import comb

# ------------------ coercion / subsampling ------------------

def _finite_float_1d(x):
    if hasattr(x, "values"):
        x = x.values
    arr = np.asarray(x, dtype=float).ravel()
    return arr[np.isfinite(arr)]

def _subsample_finite_float(x, n_target=None, rng=None, replace=False):
    if rng is None:
        rng = np.random.default_rng()
    y = _finite_float_1d(x)
    n_avail = int(y.size)
    if n_avail == 0:
        return np.asarray([], dtype=float), 0
    if (n_target is None) or (int(n_target) <= 0):
        return y, n_avail
    n_use = int(min(int(n_target), n_avail))
    if (not replace) and (n_use == n_avail):
        return y, n_avail
    idx = rng.choice(n_avail, size=n_use, replace=bool(replace))
    return y[idx], n_use

def _subsample_any(x, k_target, rng=None, replace=False, first_k=False):
    if rng is None:
        rng = np.random.default_rng()
    if hasattr(x, "values"):
        x = x.values
    arr = np.asarray(x).ravel()
    if arr.size == 0:
        return np.asarray([], dtype=arr.dtype), 0
    k_use = int(min(int(k_target), int(arr.size)))
    if k_use <= 0:
        return np.asarray([], dtype=arr.dtype), 0
    if first_k or (k_use == arr.size and not replace):
        return arr[:k_use], k_use
    idx = rng.choice(arr.size, size=k_use, replace=bool(replace))
    return arr[idx], k_use

# ------------------ gamma schedules ------------------

def gamma_1_minus_inv_sqrtn(n_eff: int, eps: float = 1e-6) -> float:
    n_eff = int(n_eff)
    if n_eff <= 0:
        return np.nan
    return float(np.clip(1.0 - n_eff**(-0.5), eps, 1.0 - eps))

def gamma_1_minus_inv_n(n_eff: int, eps: float = 1e-6) -> float:
    n_eff = int(n_eff)
    if n_eff <= 0:
        return np.nan
    return float(np.clip(1.0 - 1.0/n_eff, eps, 1.0 - eps))

def _beta_from_gamma(gamma_j: float) -> float:
    return float(np.clip(1.0 - float(gamma_j), 1e-12, 1.0))

# ------------------ multinomial loss ------------------

def kl_multinomial(p, q, eps=1e-12):
    p = np.clip(np.asarray(p, float), eps, 1.0)
    q = np.clip(np.asarray(q, float), eps, 1.0)
    return float(np.sum(p * (np.log(p) - np.log(q))))

def tv_multinomial(p, q):
    return 0.5 * float(np.sum(np.abs(np.asarray(p, float) - np.asarray(q, float))))

def l2_multinomial(p, q):
    d = np.asarray(p, float) - np.asarray(q, float)
    return float(np.dot(d, d))

def make_multinomial_loss(kind="tv"):
    if kind == "kl": return kl_multinomial
    if kind == "tv": return tv_multinomial
    if kind == "l2": return l2_multinomial
    raise ValueError(f"Unknown loss kind: {kind}")

# ------------------ radius ------------------

def multinomial_radius(n, d, beta=0.05, mode="tight_if_small_d"):
    n = int(n); d = int(d)
    if n <= 0 or d <= 1:
        return np.nan
    C0 = np.e**3 / (2*np.pi)
    if mode == "tight_if_small_d" and d <= (n * C0 / 4.0) ** (1.0 / 3.0):
        return ((d - 1) / n) * np.log(2.0 * (d - 1) / float(beta))
    return (np.log(comb(n + d - 1, d - 1)) + np.log(1.0 / float(beta))) / n

# ------------------ multinomial mapping ------------------

def _empirical_multinomial_from_levels(x_vals, levels):
    """
    x_vals: numeric responses already on the support (or at least intended to be).
    levels: np.array of allowed values
    """
    if x_vals.size == 0:
        return None
    # OpinionQA artifacts are stored on a normalized five-point support, so use tolerant matching.
    p = np.array([(np.isclose(x_vals, lv, atol=1e-8)).mean() for lv in levels], dtype=float)
    s = float(p.sum())
    if s <= 0:
        return None
    return p / s

# ------------------ KL-ball grid sup ------------------

def kl_ball_grid_centered_at_phat(p_hat, r, grid_points=5000, seed=0):
    rng = np.random.default_rng(seed)
    p_hat = np.asarray(p_hat, float)
    d = int(p_hat.size)
    if d <= 1 or (not np.isfinite(r)) or r < 0:
        return np.zeros((0, d), dtype=float)

    grid = rng.dirichlet(np.ones(d), int(grid_points))
    eps = 1e-12
    ph = np.clip(p_hat, eps, 1.0)
    P = np.clip(grid, eps, 1.0)
    kl_vals = np.sum(ph[None, :] * (np.log(ph[None, :]) - np.log(P)), axis=1)
    return grid[kl_vals <= r]

def sup_loss_over_human_kl_ball_grid(p_hat, q_hat, r, loss_kind="tv", grid_points=5000, seed=0):
    loss_fn = make_multinomial_loss(loss_kind)
    candidates = kl_ball_grid_centered_at_phat(p_hat, r, grid_points=grid_points, seed=seed)
    if candidates.shape[0] == 0:
        return loss_fn(p_hat, q_hat), p_hat.copy()
    vals = np.array([loss_fn(p_cand, q_hat) for p_cand in candidates], dtype=float)
    idx = int(np.argmax(vals))
    return float(vals[idx]), candidates[idx].copy()

# ------------------ empirical quantile curve (for plotting) ------------------

def empirical_quantile_curve(x, probs):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.full_like(np.asarray(probs, float), np.nan, dtype=float)
    xs = np.sort(x)
    p = np.clip(np.asarray(probs, float), 0.0, 1.0)
    idx = np.ceil(p * xs.size).astype(int) - 1
    idx = np.clip(idx, 0, xs.size - 1)
    return xs[idx]

# ============================================================
# A) Compute deltas in SimFidelity style: df_long + gamma_summary
# ============================================================

def compute_deltas_multinomial_opinionqa(
    full_human, simulators_dict, *,
    levels,
    n_target=None,
    k=200,
    use_adaptive_gamma=True,
    gamma_fixed=0.9,
    gamma_fn=gamma_1_minus_inv_sqrtn,   # swap to gamma_1_minus_inv_n if desired
    loss_kind="tv",
    grid_points=3000,
    seed=0,
    human_replace=False,
    model_first_k=False,
    radius_mode="tight_if_small_d",
):
    levels = np.asarray(levels, dtype=float)
    d = int(levels.size)
    if d <= 1:
        raise ValueError("levels must have length >= 2")

    # common qids
    common = set(full_human.keys())
    for sim_map in simulators_dict.values():
        common &= set(sim_map.keys())
    common = sorted(common)
    if not common:
        raise ValueError("No overlapping qids across full_human and simulators_dict.")

    rng_master = np.random.default_rng(seed)
    rows = []
    gamma_mode = "adaptive" if use_adaptive_gamma else "fixed"

    for sim_name, sim_map in simulators_dict.items():
        for qid in common:
            local_seed = int(rng_master.integers(0, 2**32 - 1))
            rng = np.random.default_rng(local_seed)

            # human: numeric and finite
            yh_sub, n_eff = _subsample_finite_float(
                full_human[qid], n_target=n_target, rng=rng, replace=human_replace
            )
            if n_eff <= 0:
                continue

            # model: may already be numeric
            ym_sub, k_eff = _subsample_any(
                sim_map[qid], k_target=k, rng=rng, replace=False, first_k=model_first_k
            )
            if k_eff <= 0:
                continue

            # coerce model to finite floats too (OpinionQA clean should be numeric)
            ym_sub_f = _finite_float_1d(ym_sub)
            if ym_sub_f.size == 0:
                continue

            # p_hat, q_hat on *exact* levels (OpinionQA should already be on support)
            p_hat = _empirical_multinomial_from_levels(yh_sub, levels=levels)
            q_hat = _empirical_multinomial_from_levels(ym_sub_f, levels=levels)
            if p_hat is None or q_hat is None:
                continue

            # gamma_j and beta_j
            if use_adaptive_gamma:
                gamma_j = float(gamma_fn(int(n_eff)))
            else:
                gamma_j = float(gamma_fixed)

            if (not np.isfinite(gamma_j)) or (gamma_j <= 0.0) or (gamma_j >= 1.0):
                continue

            beta_j = _beta_from_gamma(gamma_j)

            # radius and sup loss
            r_j = multinomial_radius(n=n_eff, d=d, beta=beta_j, mode=radius_mode)
            if not np.isfinite(r_j):
                continue

            delta_j, _ = sup_loss_over_human_kl_ball_grid(
                p_hat=p_hat, q_hat=q_hat, r=r_j,
                loss_kind=loss_kind, grid_points=grid_points, seed=local_seed
            )
            if not np.isfinite(delta_j):
                continue

            rows.append({
                "qid": qid,
                "sim": sim_name,
                "delta": float(delta_j),
                "n_eff": int(n_eff),
                "k_eff": int(k_eff),
                "gamma_j": float(gamma_j),
                "beta_j": float(beta_j),
                "r_j": float(r_j),
                "gamma_mode": gamma_mode,
            })

    df_long = pd.DataFrame(rows, columns=[
        "qid","sim","delta","n_eff","k_eff","gamma_j","beta_j","r_j","gamma_mode"
    ])

    if len(df_long) > 0:
        gamma_summary = {
            "gamma_mode": gamma_mode,
            "bar_gamma_overall": float(df_long["gamma_j"].mean()),
            "n_rows_overall": int(len(df_long)),
            "bar_gamma_by_sim": {str(k): float(v) for k, v in df_long.groupby("sim")["gamma_j"].mean().items()},
            "n_rows_by_sim": {str(k): int(v) for k, v in df_long.groupby("sim").size().items()},
        }
    else:
        gamma_summary = {
            "gamma_mode": gamma_mode,
            "bar_gamma_overall": np.nan,
            "n_rows_overall": 0,
            "bar_gamma_by_sim": {},
            "n_rows_by_sim": {},
        }

    return df_long, gamma_summary

# ============================================================
# C) Paper-style quantile fidelity plot: alpha -> Vhat(alpha)
# ============================================================

def plot_quantile_fidelity_profiles(
    df_long, *,
    sim_names=None, alpha_grid=None, color_map=None,
    linewidth=2.0, linestyle="-", legend=True,
    xlabel=r"$\alpha$", ylabel=r"$\widehat V_\ell(\alpha)$", title=None,
    figsize=(7.8, 5.2),
):
    if sim_names is None:
        sim_names = sorted(df_long["sim"].unique().tolist())

    if alpha_grid is None:
        alpha_grid = np.linspace(0.0, 1.0, 201)
    alpha_grid = np.asarray(alpha_grid, float)

    fig, ax = plt.subplots(figsize=figsize)
    out = {}

    for sim in sim_names:
        sub = df_long[df_long["sim"] == sim]
        deltas = np.asarray(sub["delta"], float)
        deltas = deltas[np.isfinite(deltas)]
        if deltas.size == 0:
            continue

        V = empirical_quantile_curve(deltas, alpha_grid)
        color = None if color_map is None else color_map.get(sim, None)
        ax.plot(alpha_grid, V, linestyle=linestyle, linewidth=linewidth, color=color, label=str(sim))
        out[str(sim)] = {"alpha": alpha_grid, "V_hat": V}

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title)
    if legend:
        ax.legend(fontsize=9, ncol=2, framealpha=0.3)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()
    return fig, out


### Final OpinionQA Plot

In [ ]:
OPINIONQA_PAPER_MODEL_ORDER = ['gpt-3.5-turbo', 'gpt-4o', 'gpt-4o-mini', 'claude-3-5-haiku', 'Llama-3.3-70B', 'Mistral-7B', 'DeepSeek-V3', 'random']
OPINIONQA_LOSS_KIND = 'tv'  # Paper setting: total variation, L(p, q) = 0.5 * ||p - q||_1

opinionqa_df_long, opinionqa_gamma_summary = compute_deltas_multinomial_opinionqa(
    full_human=real_answers,
    simulators_dict=synthetic_answers_all,
    levels=levels,
    n_target=None,
    k=100,
    use_adaptive_gamma=True,
    gamma_fn=gamma_1_minus_inv_sqrtn,
    loss_kind=OPINIONQA_LOSS_KIND,
    grid_points=2000,
    seed=42,
)

opinionqa_fig, _ = plot_quantile_fidelity_profiles(
    opinionqa_df_long,
    sim_names=OPINIONQA_PAPER_MODEL_ORDER,
    title='OpinionQA quantile fidelity profiles',
)

opinionqa_fig_path = FIGURES_DIR / 'opinionqa_quantile_fidelity_profiles.png'
opinionqa_fig.savefig(opinionqa_fig_path, dpi=200, bbox_inches='tight')
plt.close(opinionqa_fig)
print(f'Saved OpinionQA figure to: {opinionqa_fig_path}')
display(Image(filename=str(opinionqa_fig_path)))


![Figure 14 reproduction](figures/opinionqa_quantile_fidelity_profiles.png)


## Archived Figures

In [ ]:
for fig_name in [
    'eedi_quantile_fidelity_profiles.png',
    'opinionqa_quantile_fidelity_profiles.png',
]:
    fig_path = FIGURES_DIR / fig_name
    if fig_path.exists():
        display(Image(filename=str(fig_path)))
    else:
        print(f'Missing archived figure: {fig_path}')
